# Galaxy gas-loss and shock-front classification

이미 불러온 은하 이력과 cluster 이력을 함수에 전달합니다. 큰 shock 배열은 선택된 출력에서만 읽습니다. 결과는 노출·시간적 연관성의 증거이며 인과관계의 증명은 아닙니다.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
import pickle

PROJECT = Path('/Users/byeongyeonghwan/Documents/ShockFinder')
DATA = Path('/Users/byeongyeonghwan/NC_shock')
sys.path.insert(0, str(PROJECT))

from examples.galaxy_notebook import (
    DEFAULT_UNITS, prepare_saved_shock_search, analyze_saved_galaxies,
)
from examples.galaxy import (
    classify_galaxy, find_stripped_epoch, find_pericenter,
    find_front_affected_galaxies, plot_galaxy_stripping,
    plot_stripping_population,
)

with (DATA / 'dwarf_catalog_620_900.pkl').open('rb') as stream:
    sample_galaxies_history = pickle.load(stream)
with (DATA / 'cluster_info.pkl').open('rb') as stream:
    cluster_info = pickle.load(stream)
print('galaxy branches:', len(np.unique(sample_galaxies_history['last'])))


## 은하 한 개 분석

`last`는 branch의 지속 ID입니다. 아래 값은 front 3163을 확인하는 예시이며 원하는 ID로 바꿀 수 있습니다.


In [ ]:
last = 3751436
gal_branch = sample_galaxies_history[sample_galaxies_history['last'] == last]
print('history outputs:', len(gal_branch))
print('largest gas-loss onset iout:', find_stripped_epoch(gal_branch))
print('pericenters:', find_pericenter(gal_branch, cluster_info, units=DEFAULT_UNITS))


In [ ]:
search = prepare_saved_shock_search(
    gal_branch, cluster_info, DATA,
    snapshot_range=(700, 900),
    min_front_mach=3.0,
    min_front_cells=10000,
    fronts=[(785, 3163)],
    options={'pressure_scale': 'log10', 'shock_search_mode': 'events'},
)
display(pd.DataFrame(search.front_table))
print('selected outputs:', search.summary['mapped_snapshots'])


In [ ]:
analysis = classify_galaxy(
    gal_branch, search.load_product, cluster_info,
    units=DEFAULT_UNITS, options=search.options, make_plots=False,
)
events = pd.DataFrame(analysis['gas_loss_events'])
classification = pd.DataFrame(analysis['classifications'])
display(events.loc[events['significant'], [
    'galaxy_id', 'tracer', 'onset_snapshot', 'onset_time_gyr',
    'peak_time_gyr', 'end_time_gyr', 'fractional_loss',
    'peak_loss_rate_gyr', 'timescale_gyr', 'quality_flags',
]])
display(classification[[
    'galaxy_id', 'category', 'confidence', 'delta_t_shock_gyr',
    'delta_t_peri_gyr', 'merger_shock_stripping_score',
    'ordinary_rps_score', 'ambiguity_reasons',
]])


## 특정 front에 영향을 받은 은하

`include_without_loss=True`는 shock zone 겹침을 모두 보여줍니다. 기본 호출은 그 무렵 유의미한 가스 손실이 시작된 은하만 보여줍니다. 한 출력에서 겹쳤다는 사실만으로 통과나 원인을 확정하지 않습니다.


In [ ]:
all_overlaps = pd.DataFrame(find_front_affected_galaxies(
    analysis, 785, 3163, include_without_loss=True))
loss_candidates = pd.DataFrame(find_front_affected_galaxies(
    analysis, 785, 3163))
display(all_overlaps)
display(loss_candidates)


In [ ]:
fig, axes = plot_galaxy_stripping(analysis, last)
display(fig)


## 전체 은하 집단 분석

다음 셀은 전체 이력에 적용합니다. `output_dir`를 지정하면 CSV와 분석 pickle을 저장합니다. `make_plots=False`는 수백 개의 개별 그림을 자동 생성하지 않습니다. 전체 노출 비교에는 `shock_search_mode='all'`과 충분한 shock coverage가 필요합니다.


In [ ]:
population, population_search = analyze_saved_galaxies(
    sample_galaxies_history, cluster_info, DATA,
    snapshot_range=(700, 900),
    min_front_mach=3.0,
    min_front_cells=10000,
    fronts=[(785, 3163)],
    options={'pressure_scale': 'log10', 'shock_search_mode': 'events'},
    output_dir=PROJECT / 'artifacts' / 'galaxy_stripping_notebook',
    make_plots=False,
)
display(pd.DataFrame(population['classifications'])[['galaxy_id', 'category', 'confidence']].head())
print(population_search.summary)
print(population['paths'])


In [ ]:
fig, axes = plot_stripping_population(population)
display(fig)
